## 1. Importing + Constants

In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holidays
import warnings
warnings.filterwarnings('ignore')

SEED = 42
ORIGINAL_DATA_PATH = "../original-data"
PROCESSED_DATA_PATH = "../processed-data"

os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)


## 2. Preprocessing Data

In [7]:
print("1. Tải dữ liệu gốc...")
sales = pd.read_csv(f'{ORIGINAL_DATA_PATH}/sales.csv', parse_dates=['Date'])
submission = pd.read_csv(f'{ORIGINAL_DATA_PATH}/sample_submission.csv', parse_dates=['Date'])
promotions = pd.read_csv(f'{ORIGINAL_DATA_PATH}/promotions.csv', parse_dates=['start_date', 'end_date'])

print("2. Tạo trục thời gian toàn vẹn...")
# Tạo trục thời gian toàn vẹn (từ Train đến hết Test)
all_dates = pd.date_range(start=sales['Date'].min(), end=submission['Date'].max(), freq='D')
df = pd.DataFrame({'Date': all_dates})

# Đánh dấu tập Train/Test
df = df.merge(sales, on='Date', how='left')
df['Split'] = np.where(df['Date'] <= sales['Date'].max(), 'Train', 'Test')

print("3. Kỹ thuật đặc trưng Thời gian (Calendar Features)...")
def create_calendar_features(df):
    df['year'] = df['Date'].dt.year
    df['month'] = df['Date'].dt.month
    df['day'] = df['Date'].dt.day
    df['day_of_week'] = df['Date'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
    df['quarter'] = df['Date'].dt.quarter
    df['is_month_start'] = df['Date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['Date'].dt.is_month_end.astype(int)
    
    # Ngày nhận lương (Thường rơi vào đầu tháng hoặc giữa tháng)
    df['is_near_payday'] = df['day'].isin([1, 2, 14, 15, 16]).astype(int)
    
    # Mã hóa vòng (Cyclical encoding) giúp mô hình hiểu tính chu kỳ
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week']/7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week']/7)
    
    return df

df = create_calendar_features(df)

print("4. Thêm các ngày lễ Việt Nam & Khoảng cách đến Tết...")
def add_vietnam_holidays(df):
    years = df['Date'].dt.year.unique().tolist()
    vn_holidays = holidays.VN(years=years)
    
    # 1. Chuẩn hóa chuỗi và mở rộng tập từ khóa nhận diện Tết
    tet_keywords = ['tet', 'tết', 'lunar', 'new year', 'nguyên đán', 'giao thừa']
    
    # Ép kiểu toàn bộ keys về pd.Timestamp
    public_holiday_dates = [pd.to_datetime(date) for date, name in vn_holidays.items()]
    tet_dates = [
        pd.to_datetime(date) for date, name in vn_holidays.items() 
        if any(keyword in str(name).lower() for keyword in tet_keywords)
    ]
    
    # So sánh Vectorized
    df['is_public_holiday'] = df['Date'].isin(public_holiday_dates).astype(int)
    df['is_tet'] = df['Date'].isin(tet_dates).astype(int)
    
    # Tính khoảng cách đến Tết (Yếu tố quan trọng để bắt trend mua sắm trước Tết)
    def get_tet_proximity(current_date):
        distances = [(t - current_date).days for t in tet_dates]
        future_tets = [d for d in distances if d >= 0]
        past_tets = [-d for d in distances if d <= 0]
        
        days_to_tet = min(future_tets) if future_tets else 999
        days_since_tet = min(past_tets) if past_tets else 999
        return pd.Series([days_to_tet, days_since_tet])
        
    df[['days_to_tet', 'days_since_tet']] = df['Date'].apply(get_tet_proximity)
    
    # 2. Lễ thương mại Dương lịch cố định
    df['is_commercial_event'] = 0
    commercial_dates = {
        (2, 14): 'Valentine', (3, 8): 'International_Womens_Day',
        (6, 1): 'Childrens_Day', (10, 20): 'Vietnamese_Womens_Day',
        (11, 20): 'Teachers_Day', (12, 24): 'Christmas_Eve', (12, 25): 'Christmas_Day'
    }
    for (m, d), event in commercial_dates.items():
        df.loc[(df['month'] == m) & (df['day'] == d), 'is_commercial_event'] = 1

    return df

df = add_vietnam_holidays(df)

print("5. Trích xuất thông tin Khuyến mãi chuyên sâu...")
# Thay vì chỉ đếm, ta trích xuất giá trị giảm giá, thể loại, và cờ cộng dồn
def extract_promo_features(current_date):
    active = promotions[(promotions['start_date'] <= current_date) & (promotions['end_date'] >= current_date)]
    if len(active) == 0:
        return pd.Series({
            'active_promos': 0, 'total_discount_value': 0, 
            'max_discount_value': 0, 'stackable_promos': 0
        })
    return pd.Series({
        'active_promos': len(active),
        'total_discount_value': active['discount_value'].sum(),
        'max_discount_value': active['discount_value'].max(),
        'stackable_promos': active['stackable_flag'].sum()
    })

promo_features = df['Date'].apply(extract_promo_features)
df = pd.concat([df, promo_features], axis=1)

print("6. Tạo Target Lags & Rolling Windows...")
# Lags thông thường
lags = [1, 7, 14, 30, 365]
for lag in lags:
    df[f'rev_lag_{lag}'] = df['Revenue'].shift(lag)
    df[f'cogs_lag_{lag}'] = df['COGS'].shift(lag)

# Rolling Windows (Phải dùng .shift(1) để không bị rò rỉ dữ liệu của ngày hiện tại)
for window in [7, 14, 30]:
    df[f'rev_rolling_mean_{window}'] = df['Revenue'].shift(1).rolling(window).mean()
    df[f'cogs_rolling_mean_{window}'] = df['COGS'].shift(1).rolling(window).mean()
    df[f'rev_rolling_std_{window}'] = df['Revenue'].shift(1).rolling(window).std()
    df[f'cogs_rolling_std_{window}'] = df['COGS'].shift(1).rolling(window).std()

print("7. Lưu dữ liệu...")
# Loại bỏ các dòng đầu tiên bị NaN do cơ chế Lag_365
df = df[df['Date'] >= (sales['Date'].min() + pd.Timedelta(days=365))]

df.to_csv(f'{PROCESSED_DATA_PATH}/final_training_data.csv', index=False)
print(f"Kích thước bộ dữ liệu mới: {df.shape}")

1. Tải dữ liệu gốc...
2. Tạo trục thời gian toàn vẹn...
3. Kỹ thuật đặc trưng Thời gian (Calendar Features)...
4. Thêm các ngày lễ Việt Nam & Khoảng cách đến Tết...
5. Trích xuất thông tin Khuyến mãi chuyên sâu...
6. Tạo Target Lags & Rolling Windows...
7. Lưu dữ liệu...
Kích thước bộ dữ liệu mới: (4016, 48)
